In [1]:
import tensorflow as tf
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

2026-09-21 14:47:26.972737: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-21 14:47:26.999314: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-21 14:47:26.999340: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-21 14:47:27.000383: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-21 14:47:27.006683: I tensorflow/core/platform/cpu_feature_guar

TF: 2.15.1
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-09-21 14:47:28.220375: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-21 14:47:28.267495: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-21 14:47:28.267526: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable(package="Custom")
class RadialWaveEvolutionaryConv(layers.Layer):

    def __init__(self,
                 num_patches=4,
                 num_feature_maps=32,
                 kernel_size=3,
                 **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.num_feature_maps = num_feature_maps
        self.kernel_size = kernel_size

    def build(self, input_shape):

        _, H, W, C = input_shape

        self.H = H
        self.W = W

        self.patch_h = H // self.num_patches
        self.patch_w = W // self.num_patches

        # Decision projection
        self.decision_filters = self.add_weight(
            shape=(C, self.num_feature_maps),
            initializer=tf.keras.initializers.GlorotNormal(),
            trainable=True,
            name="decision_space_filters"
        )

        # Convolution kernel
        self.gaussian_kernel = self.add_weight(
            shape=(self.kernel_size,
                   self.kernel_size,
                   C,
                   self.num_feature_maps),
            initializer=tf.keras.initializers.HeNormal(),
            trainable=True,
            name="gaussian_mutation_kernel"
        )

        # Learnable sigma (no user input)
        self.sigma = self.add_weight(
            shape=(),
            initializer=tf.keras.initializers.Constant(0.1),
            trainable=True,
            name="wave_sigma"
        )

        super().build(input_shape)

    # --------------------------------------------------
    # Pareto (tanh, batch-safe)
    # --------------------------------------------------
    def soft_pareto_score(self, objectives):
        # [B, N, M]

        f_i = tf.expand_dims(objectives, axis=2)  # [B, N, 1, M]
        f_j = tf.expand_dims(objectives, axis=1)  # [B, 1, N, M]

        domination = 0.5 * (1.0 + tf.nn.tanh(f_j - f_i))

        domination = tf.clip_by_value(domination, 1e-6, 1.0)

        domination_prod = tf.reduce_prod(domination, axis=-1)  # [B, N, N]

        score = tf.reduce_sum(domination_prod, axis=-1)  # [B, N]

        return score

    # --------------------------------------------------
    # Call
    # --------------------------------------------------
    def call(self, inputs):

        B = tf.shape(inputs)[0]
        C = tf.shape(inputs)[-1]

        # Patch extraction
        patches = tf.image.extract_patches(
            images=inputs,
            sizes=[1, self.patch_h, self.patch_w, 1],
            strides=[1, self.patch_h, self.patch_w, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        num_total_patches = self.num_patches * self.num_patches
        patch_area = self.patch_h * self.patch_w

        patches = tf.reshape(
            patches,
            [B, num_total_patches, patch_area, C]
        )

        patches_mean = tf.reduce_mean(patches, axis=2)  # [B, N, C]

        # Objectives
        objectives = tf.matmul(patches_mean, self.decision_filters)  # [B, N, M]

        # Pareto
        pareto_score = self.soft_pareto_score(objectives)

        weights = tf.nn.softmax(-pareto_score, axis=-1)
        weights = tf.expand_dims(weights, -1)

        weighted_objectives = objectives * weights

        # Radial wave
        center = tf.reduce_mean(weighted_objectives, axis=1, keepdims=True)

        radial_distance = tf.norm(weighted_objectives - center, axis=-1)

        sigma = tf.nn.softplus(self.sigma) + 1e-6

        wave = tf.math.special.bessel_j0(radial_distance)
        decay = tf.exp(- (radial_distance ** 2) / (2.0 * sigma ** 2))

        wave = wave * decay  # [B, N]

        # Expand to spatial
        wave = tf.reshape(wave, [B, self.num_patches, self.num_patches, 1])

        wave = tf.image.resize(
            wave,
            size=(self.H, self.W),
            method="nearest"
        )

        # Convolution
        conv_out = tf.nn.conv2d(
            inputs,
            self.gaussian_kernel,
            strides=1,
            padding="SAME"
        )

        return conv_out * wave

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "num_feature_maps": self.num_feature_maps,
            "kernel_size": self.kernel_size
        })
        return config

In [3]:
def conv_block(x,
               filters,
               kernel_size=3,
               activation="tanh",
               padding="same",
               decoder=False,
               num_patches=4,
               overlap_ratio=0.02,
               use_overlap=False,
               dilation_rate=2):

    res = x

    # ============================================================
    # ENCODER BLOCK
    # ============================================================
    if not decoder:


        # -------------------------------
        # Branch 2: Evolutionary Conv
        # -------------------------------
        x1 = RadialWaveEvolutionaryConv(
            num_patches=num_patches,
            num_feature_maps=filters,
            kernel_size=kernel_size
        )(x)

        x1 = layers.BatchNormalization()(x1)
        x1 = layers.Activation(activation)(x1)

        # -------------------------------
        # Depthwise Patch Attention
        # -------------------------------

        # Residual projection
        res = layers.Conv2D(filters, 1, padding=padding)(res)

        # Final fusion
        out = layers.Add()([res, x1])

        return out

    # ============================================================
    # DECODER BLOCK
    # ============================================================
    else:

        x1 = RadialWaveEvolutionaryConv(
            num_patches=num_patches,
            num_feature_maps=filters,
            kernel_size=kernel_size
        )(x)

        x1 = layers.BatchNormalization()(x1)
        x1 = layers.Activation(activation)(x1)

        return x1

In [4]:
import tensorflow as tf
from tensorflow.keras import layers

# ---------------------------------------------------------
# Geodesic Kernel Downsampler
# ---------------------------------------------------------

@tf.keras.utils.register_keras_serializable(package="Custom")
class RegionawareAnistropicMeanPool2D(layers.Layer):

    def __init__(
        self,
        pool_size=2,
        geo_iters=3,
        geo_temp=1.0,
        robust_kernel="huber",
        tukey_c=4.685,
        huber_delta=1.0,
        cauchy_alpha=1.0,
        geman_lambda=1.0,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.pool_size = pool_size
        self.geo_iters = geo_iters
        self.geo_temp = geo_temp
        self.robust_kernel = robust_kernel.lower()

        # kernel params
        self.huber_delta = huber_delta
        self.tukey_c = tukey_c
        self.cauchy_alpha = cauchy_alpha
        self.geman_lambda = geman_lambda

    def build(self, input_shape):
        c = input_shape[-1]
        self.metric_raw = self.add_weight(
            shape=(c,),
            initializer=tf.keras.initializers.Constant(1.0),
            trainable=True,
            name="metric_diag_unconstrained"
        )
        super().build(input_shape)

    # ---------------- robust weighting functions ----------------
    def _robust_weights(self, dist_sq):
        eps = 1e-9

        if self.robust_kernel == "huber":
            delta = self.huber_delta
            mask = dist_sq <= delta**2
            w_quad = 1.0
            w_lin = delta / (tf.sqrt(dist_sq) + eps)
            return tf.where(mask, w_quad, w_lin)

        if self.robust_kernel == "tukey":
            c = self.tukey_c
            mask = dist_sq <= c**2
            z = dist_sq / (c**2 + eps)
            w = (1 - z)**2
            return tf.where(mask, w, tf.zeros_like(dist_sq))

        if self.robust_kernel == "cauchy":
            a = self.cauchy_alpha
            return 1.0 / (1.0 + dist_sq / (a**2) + eps)

        if self.robust_kernel == "geman":
            lam = self.geman_lambda
            return 1.0 / ((1.0 + dist_sq / (lam + eps))**2)

        return tf.nn.softmax(-dist_sq, axis=-1)

    # ---------------- main pooling routine ----------------
    def call(self, x):
        if isinstance(self.pool_size, int):
            ph = pw = self.pool_size
        else:
            ph, pw = self.pool_size

        b, h, w, c = tf.unstack(tf.shape(x))
        pad_h = tf.math.floormod(-h, ph)
        pad_w = tf.math.floormod(-w, pw)
        x_pad = tf.pad(x, [[0,0],[0,pad_h],[0,pad_w],[0,0]])

        Hp = tf.shape(x_pad)[1] // ph
        Wp = tf.shape(x_pad)[2] // pw

        ksizes = [1, ph, pw, 1]
        strides = [1, ph, pw, 1]
        patches = tf.image.extract_patches(
            images=x_pad,
            sizes=ksizes,
            strides=strides,
            rates=[1,1,1,1],
            padding="VALID"
        )

        P = ph * pw
        patches = tf.reshape(patches, [b, Hp, Wp, P, c])

        metric = tf.nn.softplus(self.metric_raw) + 1e-6
        mean = tf.reduce_mean(patches, axis=3)

        for _ in range(self.geo_iters):
            diff = patches - tf.expand_dims(mean, axis=3)
            dist_sq = tf.reduce_sum(diff * diff * tf.reshape(metric, [1,1,1,1,c]), axis=-1)
            w = self._robust_weights(dist_sq)
            w = tf.nn.softmax(w / self.geo_temp, axis=-1)
            w = tf.expand_dims(w, axis=-1)
            mean = tf.reduce_sum(patches * w, axis=3)

        return mean

    def get_config(self):
        base = super().get_config()
        base.update({
            "pool_size": self.pool_size,
            "geo_iters": self.geo_iters,
            "geo_temp": self.geo_temp,
            "robust_kernel": self.robust_kernel,
            "tukey_c": self.tukey_c,
            "huber_delta": self.huber_delta,
            "cauchy_alpha": self.cauchy_alpha,
            "geman_lambda": self.geman_lambda
        })
        return base



In [5]:
def build_prior_guided_unet(input_shape=(256, 256, 3), base_filters=64, num_classes=1):
    inputs = layers.Input(shape=input_shape)

    # ===================== Encoder =====================

    # -------- Level 1 --------
    f1 = conv_block(inputs, base_filters)
    s1 = RegionawareAnistropicMeanPool2D()(f1)

    p1 = layers.Dropout(0.05)(s1)
    p1 = layers.GaussianDropout(0.1)(p1)

    # -------- Level 2 --------
    f2 = conv_block(p1, base_filters * 2)
    s2 = RegionawareAnistropicMeanPool2D()(f2)

    p2 = layers.Dropout(0.15)(s2)
    p2 = layers.GaussianDropout(0.2)(p2)

    # -------- Level 3 --------
    f3 = conv_block(p2, base_filters * 4)
    s3 = RegionawareAnistropicMeanPool2D()(f3)

    p3 = layers.Dropout(0.2)(s3)

    # -------- Level 4 --------
    f4 = conv_block(p3, base_filters * 8)
    s4 = RegionawareAnistropicMeanPool2D()(f4)

    p4 = layers.Dropout(0.25)(s4)
    p4 = layers.GaussianDropout(0.2)(p4)

    # -------- Bottleneck --------
    f5 = conv_block(p4, base_filters * 12)
    f5 = layers.SpatialDropout2D(0.2)(f5)
    f5 = layers.GaussianDropout(0.1)(f5)

    # -------- Up 4 --------
    d4 = layers.UpSampling2D(size=2, interpolation="bilinear")(f5)
    d4 = layers.Dropout(0.05)(d4)
    d4 = layers.GaussianDropout(0.18)(d4)
    d4 = conv_block(d4, base_filters * 8, decoder=True)
    d4 = layers.Add()([f4, d4])

    # -------- Up 3 --------
    d3 = layers.UpSampling2D(size=2, interpolation="bilinear")(d4)
    d3 = layers.GaussianDropout(0.18)(d3)
    d3 = layers.Dropout(0.1)(d3)
    d3 = conv_block(d3, base_filters * 4, decoder=True)
    d3 = layers.Add()([f3, d3])

    # -------- Up 2 --------
    d2 = layers.UpSampling2D(size=2, interpolation="bilinear")(d3)
    d2 = conv_block(d2, base_filters * 2, decoder=True)
    d2 = layers.Dropout(0.22)(d2)
    d2 = layers.Add()([f2, d2])

    # -------- Up 1 --------
    d1 = layers.UpSampling2D(size=2, interpolation="bilinear")(d2)
    d1 = conv_block(d1, base_filters, decoder=True)
    d1 = layers.Dropout(0.24)(d1)

    # ===================== Output =====================
    fused = layers.Add()([f1, d1])
    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid")(fused)

    model = tf.keras.Model(
        inputs,
        outputs,
        name="ResidualUNet_Deep5_UpSampling"
    )

    return model

In [6]:
model = build_prior_guided_unet(input_shape=(256,256, 3), base_filters=32)
model.summary()

2026-09-21 14:47:28.310105: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-21 14:47:28.310163: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-21 14:47:28.310177: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-21 14:47:28.414586: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-21 14:47:28.414650: I external/local_xla/xla/stream_executor

Model: "ResidualUNet_Deep5_UpSampling"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 radial_wave_evolutionary_c  (None, 256, 256, 32)         961       ['input_1[0][0]']             
 onv (RadialWaveEvolutionar                                                                       
 yConv)                                                                                           
                                                                                                  
 batch_normalization (Batch  (None, 256, 256, 32)         128       ['radial_wave_evolutionary_con
 Normalization)                                                     v[

In [7]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras import backend as K

# -----------------------------
# Optional imports / fallbacks
# -----------------------------
try:
    from tensorflow.keras.optimizers import AdamW
except Exception:
    AdamW = tf.keras.optimizers.Adam  # fallback to Adam if AdamW not present

# -----------------------------
# Metrics & simple losses
# -----------------------------
@register_keras_serializable(package="Custom")
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    denominator = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])
    dice = (2. * intersection + smooth) / (denominator + smooth)
    return tf.reduce_mean(dice)

@register_keras_serializable(package="Custom")
def dice_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - dice_coef(y_true, y_pred, smooth)

@register_keras_serializable(package="Custom")
def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=[1,2,3])
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)

@register_keras_serializable(package="Custom")
def iou_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - iou_coef(y_true, y_pred, smooth)

@register_keras_serializable(package="Custom")
def aggregated_jaccard_index(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=[1, 2, 3])
    jaccard = (intersection + smooth) / (union + smooth)

    weights = tf.reduce_sum(y_true, axis=[1, 2, 3]) + smooth
    weighted_mean = tf.reduce_sum(jaccard * weights) / tf.reduce_sum(weights)
    return weighted_mean

@register_keras_serializable(package="Custom")
def f1_score(y_true, y_pred, threshold=0.5, smooth=1e-7):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    true_pos = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    predicted_pos = tf.reduce_sum(y_pred, axis=[1,2,3])
    possible_pos = tf.reduce_sum(y_true, axis=[1,2,3])

    precision = (true_pos + smooth) / (predicted_pos + smooth)
    recall = (true_pos + smooth) / (possible_pos + smooth)
    f1 = 2 * (precision * recall) / (precision + recall + smooth)
    return tf.reduce_mean(f1)

In [8]:
# -----------------------------
# Serializable metrics wrappers
# -----------------------------
@register_keras_serializable(package="Custom")
class IoUMetric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="iou_coef"):
        super().__init__(iou_coef, name=name)

@register_keras_serializable(package="Custom")
class DiceMetric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="dice_coef"):
        super().__init__(dice_coef, name=name)

@register_keras_serializable(package="Custom")
class F1Metric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="f1_score"):
        super().__init__(f1_score, name=name)

@register_keras_serializable(package="Custom")
class AJIMetric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="aggregated_jaccard_index"):
        super().__init__(aggregated_jaccard_index, name=name)

In [9]:
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable

# -----------------------------
# Simplified LambdaLogger callback
# -----------------------------
@register_keras_serializable(package="Custom")
class LambdaLogger(tf.keras.callbacks.Callback):
    def __init__(self, writer=None):
        super().__init__()
        self.writer = writer

    def on_epoch_end(self, epoch, logs=None):
        if not hasattr(self.model, "lambda1"):
            return

        # Fetch softplus-transformed adaptive weights
        l1 = tf.nn.softplus(self.model.lambda1).numpy()
        l2 = tf.nn.softplus(self.model.lambda2).numpy()
        l3 = tf.nn.softplus(self.model.lambda3).numpy()
        lfg = tf.nn.softplus(self.model.lambda_fg).numpy()
        lbg = tf.nn.softplus(self.model.lambda_bg).numpy()

        # Print clean epoch summary
        msg = (f"Epoch {epoch+1}: λ1={l1:.4f}, λ2={l2:.4f}, λ3={l3:.4f}, "
               f"λ_fg={lfg:.4f}, λ_bg={lbg:.4f}")
        print(msg)

        # Log to TensorBoard if writer is available
        if self.writer is not None:
            with self.writer.as_default():
                tf.summary.scalar("lambda1", l1, step=epoch)
                tf.summary.scalar("lambda2", l2, step=epoch)
                tf.summary.scalar("lambda3", l3, step=epoch)
                tf.summary.scalar("lambda_fg", lfg, step=epoch)
                tf.summary.scalar("lambda_bg", lbg, step=epoch)
                self.writer.flush()

    def get_config(self):
        return {"writer": None}


In [10]:
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable(package="Custom")
class HyperStructuralLoss(tf.keras.losses.Loss):
    """
    Segmentation loss with learnable λ weights.

    Components:
        L_seg = λ2 * Dice + λ3 * IoU

    NOTE: the BCE term, the canvas consistency loss, the patchwise
    discriminator loss, and the center/boundary structural loss
    have all been removed. Only Dice + IoU remain.
    """
    def __init__(self, init_lambdas=None, name="hyper_structural_loss"):
        super().__init__(name=name)
        if init_lambdas is None:
            init_lambdas = {"l2": 3.0, "l3": 4.5}

        # Learnable λ variables
        self.l2 = tf.Variable(init_lambdas["l2"], trainable=True, dtype=tf.float32, name="l2")
        self.l3 = tf.Variable(init_lambdas["l3"], trainable=True, dtype=tf.float32, name="l3")

        # Explicitly expose trainable weights for TensorFlow
        self._trainable_weights = [self.l2, self.l3]

    def call(self, y_true, y_pred):
        """
        Args:
            y_true: ground-truth segmentation mask
            y_pred: predicted segmentation mask
        """
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        # --- Segmentation Loss ---
        L2, L3 = tf.nn.softplus(self.l2), tf.nn.softplus(self.l3)
        intersection = tf.reduce_sum(y_true * y_pred)
        dice = 1.0 - (2.0 * intersection + 1e-6) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-6)
        union = tf.reduce_sum(y_true + y_pred) - intersection
        iou = 1.0 - (intersection + 1e-6) / (union + 1e-6)
        seg_loss = L2 * dice + L3 * iou

        return seg_loss

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "init_lambdas": {
                "l2": float(self.l2.numpy()),
                "l3": float(self.l3.numpy())
            }
        })
        return cfg

In [11]:
from tensorflow.keras.optimizers import AdamW
from datetime import datetime

In [12]:
import tensorflow as tf
from glob import glob
from PIL import Image
import numpy as np
import os

# Parameters
img_size = 256
batch_size = 4

# Paths
train_img_paths = sorted(glob("../dataset_split/augmented_train/images/*.png"))
train_mask_paths = sorted(glob("../dataset_split/augmented_train/masks/*.png"))

val_img_paths = sorted(
    glob("../dataset_split/val/images/*.png") +
    glob("../dataset_split/val/images/*.tif") +
    glob("../dataset_split/val/images/*.bmp")
)
val_mask_paths = sorted(glob("../dataset_split/val/masks/*.bmp"))

print("Train images:", len(train_img_paths))
print("Train masks:", len(train_mask_paths))
print("Val images:", len(val_img_paths))
print("Val masks:", len(val_mask_paths))

# -----------------------------
# STEM HELPERS
# -----------------------------
def stem(p):
    return os.path.splitext(os.path.basename(p))[0]

def mask_stem(p):
    s = os.path.splitext(os.path.basename(p))[0]
    return s.replace("_lesion", "")  # adjust if your suffix differs (case/hyphen)

# -----------------------------
# ALIGN TRAIN PAIRS BY STEM
# -----------------------------
train_img_dict = {stem(p): p for p in train_img_paths}
train_mask_dict = {mask_stem(p): p for p in train_mask_paths}

train_common = sorted(set(train_img_dict) & set(train_mask_dict))
train_missing_masks = set(train_img_dict) - set(train_mask_dict)
train_missing_imgs = set(train_mask_dict) - set(train_img_dict)

if train_missing_masks:
    print(f"⚠️ {len(train_missing_masks)} train images have no matching mask, e.g.:", list(train_missing_masks)[:5])
if train_missing_imgs:
    print(f"⚠️ {len(train_missing_imgs)} train masks have no matching image, e.g.:", list(train_missing_imgs)[:5])

train_img_paths = [train_img_dict[s] for s in train_common]
train_mask_paths = [train_mask_dict[s] for s in train_common]

assert len(train_img_paths) == len(train_mask_paths) == len(train_common), "❌ Train alignment failed"
print("✅ Aligned train pairs:", len(train_common))

# -----------------------------
# ALIGN VAL PAIRS BY STEM
# -----------------------------
val_img_dict = {stem(p): p for p in val_img_paths}
val_mask_dict = {mask_stem(p): p for p in val_mask_paths}

val_common = sorted(set(val_img_dict) & set(val_mask_dict))
val_missing_masks = set(val_img_dict) - set(val_mask_dict)
val_missing_imgs = set(val_mask_dict) - set(val_img_dict)

if val_missing_masks:
    print(f"⚠️ {len(val_missing_masks)} val images have no matching mask, e.g.:", list(val_missing_masks)[:5])
if val_missing_imgs:
    print(f"⚠️ {len(val_missing_imgs)} val masks have no matching image, e.g.:", list(val_missing_imgs)[:5])

val_img_paths = [val_img_dict[s] for s in val_common]
val_mask_paths = [val_mask_dict[s] for s in val_common]

assert len(val_img_paths) == len(val_mask_paths) == len(val_common), "❌ Val alignment failed"
print("✅ Aligned val pairs:", len(val_common))

# -----------------------------
# LOADER
# -----------------------------
def load_image_mask(img_path, mask_path):
    img = np.array(
        Image.open(img_path.numpy().decode("utf-8"))
        .convert("RGB")
        .resize((img_size, img_size), Image.BILINEAR)
    ) / 255.0
    mask = np.array(
        Image.open(mask_path.numpy().decode("utf-8"))
        .convert("L")
        .resize((img_size, img_size), Image.NEAREST)
    )
    mask = (mask > 127).astype(np.float32)[..., None]
    return img.astype(np.float32), mask

def tf_load_image_mask(img_path, mask_path):
    img, mask = tf.py_function(
        load_image_mask, [img_path, mask_path], [tf.float32, tf.float32]
    )
    img.set_shape([img_size, img_size, 3])
    mask.set_shape([img_size, img_size, 1])
    return img, mask

# -----------------------------
# DATASETS
# -----------------------------
train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_img_paths, train_mask_paths))
    .map(tf_load_image_mask, num_parallel_calls=tf.data.AUTOTUNE).cache()
    .shuffle(buffer_size=len(train_img_paths), seed=42)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((val_img_paths, val_mask_paths))
    .map(tf_load_image_mask, num_parallel_calls=tf.data.AUTOTUNE).cache()
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_dataset).numpy())

Train images: 720
Train masks: 720
Val images: 16
Val masks: 16
✅ Aligned train pairs: 720
✅ Aligned val pairs: 16


2026-09-21 14:47:29.184639: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Training batches: 180
Validation batches: 4


In [13]:
class StructTrainer(tf.keras.Model):
    def __init__(self, backbone, loss_fn, optimizer, metrics):
        super().__init__()
        self.backbone = backbone
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.metrics_list = metrics

        # Expose learnable λ weights for monitoring
        for key in ["l2", "l3"]:
            setattr(self, key, getattr(loss_fn, key))

    def compile(self):
        super().compile(optimizer=self.optimizer)

    def train_step(self, data):
        x, y_true = data
        with tf.GradientTape() as tape:
            y_pred = self.backbone(x, training=True)
            loss = self.loss_fn(y_true, y_pred)

        grads = tape.gradient(loss, self.backbone.trainable_variables + self.loss_fn._trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.backbone.trainable_variables + self.loss_fn._trainable_weights))

        logs = {"loss": loss}
        for m in self.metrics_list:
            m.update_state(y_true, y_pred)
            logs[m.name] = m.result()
        return logs

    def test_step(self, data):
        x, y_true = data
        y_pred = self.backbone(x, training=False)
        loss = self.loss_fn(y_true, y_pred)

        logs = {"val_loss": loss}
        for m in self.metrics_list:
            m.update_state(y_true, y_pred)
            logs[f"val_{m.name}"] = m.result()
        return logs

In [14]:

optimizer = AdamW(learning_rate=1e-3, weight_decay=1e-10)

# 3️⃣ Metrics
metrics_list = [
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    #tf.keras.metrics.MeanIoU(num_classes=2, name="iou"),
    IoUMetric(), DiceMetric(), F1Metric(), AJIMetric()
]

# 4️⃣ Loss
loss_fn = HyperStructuralLoss()
ll=LambdaLogger()
# 5️⃣ Trainer
seg_model = StructTrainer(model, loss_fn, optimizer, metrics_list)
seg_model.compile()

# 6️⃣ Callbacks
log_dir = "./logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")
writer = tf.summary.create_file_writer(log_dir)
lambda_logger = LambdaLogger(writer=writer)

early_stp = tf.keras.callbacks.EarlyStopping(
    monitor='val_val_iou_coef', patience=35, mode='max', restore_best_weights=True
)

lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_val_iou_coef', mode='max', factor=0.1, patience=10, min_lr=1e-5, verbose=1
)

tb = tf.keras.callbacks.TensorBoard(log_dir=log_dir)

class SaveBackboneCallback(tf.keras.callbacks.Callback):
    def __init__(self, save_path):
        super().__init__()
        self.save_path = save_path
        self.best_iou = 0.0

    def on_epoch_end(self, epoch, logs=None):
        val_iou = logs.get("val_val_iou_coef", 0)
        if val_iou >= self.best_iou:
            self.best_iou = val_iou
            self.model.backbone.save(self.save_path)
            print(f"\n✅ Saved improved backbone at epoch {epoch+1} with val_iou={val_iou:.4f}")

save_backbone = SaveBackboneCallback("prior_gided_unet.keras")

# 7️⃣ Training
history = seg_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=150,
    callbacks=[early_stp, lr_callback, tb, lambda_logger, save_backbone],batch_size=4,
    verbose=1
)
print("\n🎯 Training complete. Backbone saved and ready for inference.")


Epoch 1/150


2026-09-21 14:47:36.084333: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inResidualUNet_Deep5_UpSampling/spatial_dropout2d/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-09-21 14:47:40.237759: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2026-09-21 14:47:40.361928: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-09-21 14:47:42.801885: I external/local_xla/xla/service/service.cc:168] XLA service 0x7ee5f6b32840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-09-21 14:47:42.801920: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Ti, Compute Capability 8.9
2026-09-21 14:47:42.805974: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:

180/180 [==============================] - ETA: 0s - loss: 2.8612 - accuracy: 0.7913 - iou_coef: 0.5464 - dice_coef: 0.6822 - f1_score: 0.6874 - aggregated_jaccard_index: 0.6039
✅ Saved improved backbone at epoch 1 with val_iou=0.4081
180/180 [==============================] - 41s 129ms/step - loss: 2.8577 - accuracy: 0.7913 - iou_coef: 0.5464 - dice_coef: 0.6822 - f1_score: 0.6874 - aggregated_jaccard_index: 0.6039 - val_val_loss: 4.5184 - val_val_accuracy: 0.7949 - val_val_iou_coef: 0.4081 - val_val_dice_coef: 0.5702 - val_val_f1_score: 0.5827 - val_val_aggregated_jaccard_index: 0.3875 - lr: 0.0010
Epoch 2/150
180/180 [==============================] - ETA: 0s - loss: 2.3614 - accuracy: 0.8434 - iou_coef: 0.6150 - dice_coef: 0.7398 - f1_score: 0.7451 - aggregated_jaccard_index: 0.6447
✅ Saved improved backbone at epoch 2 with val_iou=0.5878
180/180 [==============================] - 19s 106ms/step - loss: 2.3580 - accuracy: 0.8434 - iou_coef: 0.6150 - dice_coef: 0.7398 - f1_score: 0.

In [15]:
from glob import glob
import os
import tensorflow as tf
import numpy as np
from PIL import Image
# =========================================================
# CONFIG
# =========================================================
IMG_DIR = "../dataset_split/test/images"
MSK_DIR = "../dataset_split/test/masks"
img_size   = 256
batch_size = 38
AUTOTUNE = tf.data.AUTOTUNE

# =========================================================
# SAFE FILE COLLECTION
# =========================================================
def valid_name(p):
    return "Zone" not in os.path.basename(p)

image_paths = sorted([
    p for p in glob(os.path.join(IMG_DIR, "*"))
    if valid_name(p)
])
mask_paths = sorted([
    p for p in glob(os.path.join(MSK_DIR, "*"))
    if valid_name(p)
])

print("Images found:", len(image_paths))
print("Masks found :", len(mask_paths))

# =========================================================
# STEM HELPERS
# Masks are named "<imagename>_lesion.<ext>", so strip that
# suffix to recover the shared base name.
# =========================================================
MASK_SUFFIX = "_lesion"

def image_stem(p):
    return os.path.splitext(os.path.basename(p))[0]

def mask_stem(p):
    name = os.path.splitext(os.path.basename(p))[0]
    if name.endswith(MASK_SUFFIX):
        name = name[: -len(MASK_SUFFIX)]
    return name

# =========================================================
# PAIR IMAGES WITH MASKS BY BASE NAME
# (robust to any ordering mismatch, not just sorted-order luck)
# =========================================================
mask_lookup = {mask_stem(p): p for p in mask_paths}

paired_images = []
paired_masks = []
missing = []

for img_p in image_paths:
    key = image_stem(img_p)
    if key in mask_lookup:
        paired_images.append(img_p)
        paired_masks.append(mask_lookup[key])
    else:
        missing.append(img_p)

if missing:
    raise AssertionError(
        f"{len(missing)} image(s) have no matching mask, e.g.: {missing[:5]}"
    )

image_paths = paired_images
mask_paths = paired_masks

print("Paired samples:", len(image_paths))

# =========================================================
# NUMPY LOADER
# =========================================================
def _load_numpy(img_path, mask_path):
    img_path  = img_path.numpy().decode("utf-8")
    mask_path = mask_path.numpy().decode("utf-8")

    # -----------------------------------------------------
    # IMAGE
    # -----------------------------------------------------
    img = Image.open(img_path).convert("RGB")
    img = img.resize(
        (img_size, img_size),
        Image.BILINEAR,
    )
    img = np.array(img, dtype=np.float32) / 255.0

    # -----------------------------------------------------
    # MASK
    # -----------------------------------------------------
    mask = Image.open(mask_path).convert("L")
    mask = mask.resize(
        (img_size, img_size),
        Image.NEAREST,
    )
    mask = np.array(mask, dtype=np.float32)

    # Binary mask
    mask = (mask > 127).astype(np.float32)
    # Add channel dim
    mask = np.expand_dims(mask, axis=-1)

    return img, mask

# =========================================================
# TF WRAPPER
# =========================================================
def load_image_mask(img_path, mask_path):
    img, mask = tf.py_function(
        _load_numpy,
        [img_path, mask_path],
        [tf.float32, tf.float32],
    )
    img.set_shape([img_size, img_size, 3])
    mask.set_shape([img_size, img_size, 1])
    targets = {
        "segmentation": mask
    }
    return img, targets

# =========================================================
# DATASET
# =========================================================
test_dataset = (
    tf.data.Dataset
    .from_tensor_slices((image_paths, mask_paths))
    .map(
        load_image_mask,
        num_parallel_calls=AUTOTUNE,
    )
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

# =========================================================
# DEBUG
# =========================================================
print(
    "Test batches:",
    tf.data.experimental.cardinality(test_dataset).numpy()
)

# =========================================================
# OPTIONAL SANITY CHECK
# =========================================================
for batch_images, batch_targets in test_dataset.take(1):
    print("\nImages Shape:")
    print(batch_images.shape)
    print("\nSegmentation Shape:")
    print(batch_targets["segmentation"].shape)

Images found: 40
Masks found : 40
Paired samples: 40
Test batches: 2

Images Shape:
(38, 256, 256, 3)

Segmentation Shape:
(38, 256, 256, 1)


In [16]:
import numpy as np
from sklearn.metrics import f1_score as sk_f1_score, accuracy_score

# --- Initialize metric lists ---
ious, dices, accuracies, f1s, ajis = [], [], [], [], []

# --- Evaluation Loop ---
for imgs, masks in test_dataset:
    # Forward pass
    masks = masks["segmentation"]  # (B, 256, 256, 1)

    preds = model.predict(imgs, verbose=0)

    # Handle multi-output model (segmentation, centers, boundaries)
    if isinstance(preds, (list, tuple)):
        seg_preds = preds[0]       # segmentation output
        # center_maps = preds[1]   # not used for metrics
        # boundary_maps = preds[2] # not used for metrics
    else:
        seg_preds = preds

    # --- Threshold segmentation ---
    preds_bin = (seg_preds > 0.5).astype("float32")

    # --- Convert tensors to numpy safely ---
    masks_np = masks.numpy() if isinstance(masks, tf.Tensor) else masks
    preds_np = preds_bin.astype(np.float32)

    # Ensure shapes match
    if masks_np.shape != preds_np.shape:
        preds_np = tf.image.resize(preds_np, masks_np.shape[1:3]).numpy()

    # --- Flatten for sklearn metrics ---
    masks_flat = masks_np.reshape(-1)
    preds_flat = preds_np.reshape(-1)

    # --- Ensure binary (for safety) ---
    masks_flat = np.round(masks_flat).astype(int)
    preds_flat = np.round(preds_flat).astype(int)

    # --- Tensor-based Metrics ---
    iou_val = iou_coef(masks_np, preds_np).numpy()
    dice_val = dice_coef(masks_np, preds_np).numpy()
    aji_val = aggregated_jaccard_index(masks_np, preds_np).numpy()

    ious.append(iou_val)
    dices.append(dice_val)
    ajis.append(aji_val)

    # --- Classical sklearn metrics ---
    acc = accuracy_score(masks_flat, preds_flat)
    f1 = sk_f1_score(masks_flat, preds_flat, zero_division=1)

    accuracies.append(acc)
    f1s.append(f1)

# --- Compute mean metrics ---
mean_iou = np.mean(ious)
mean_dice = np.mean(dices)
mean_aji = np.mean(ajis)
mean_accuracy = np.mean(accuracies)
mean_f1 = np.mean(f1s)

# --- Print results ---
print("\n===== Segmentation Evaluation =====")
print(f"Mean IoU:              {mean_iou:.4f}")
print(f"Mean Dice Coefficient: {mean_dice:.4f}")
print(f"Mean AJI:              {mean_aji:.4f}")
print(f"Mean Accuracy:         {mean_accuracy:.4f}")
print(f"Mean F1 Score:         {mean_f1:.4f}")


2026-09-21 15:25:27.275153: W external/local_tsl/tsl/framework/bfc_allocator.cc:366] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.



===== Segmentation Evaluation =====
Mean IoU:              0.9208
Mean Dice Coefficient: 0.9573
Mean AJI:              0.9289
Mean Accuracy:         0.9777
Mean F1 Score:         0.9618
